In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import tensorflow as tf

In [2]:
import numpy as np

def same_padding(image, kernel):
    
    row, col = image.shape
    kernel_row, kernel_col = kernel.shape

    output_row, output_col = row + kernel_row - 1, col + kernel_col - 1
    img = np.zeros(shape=(output_row, output_col) )
    ans = np.zeros(shape=(row, col))
    
    for i in range(row):
        for j in range(col):
            img[ i + ( kernel_row - 1) // 2, j + ( kernel_col - 1) // 2 ] = image[i, j] 

    for i in range(row):
        for j in range(col):
            ans[i, j] = np.sum( img[i: i + kernel_row, j : j + kernel_col ] * kernel ) 

    return img, ans

def valid_padding(image, kernel):

    row, col = image.shape
    kernel_row, kernel_col = kernel.shape

    output_row, output_col = row - kernel_row + 1, col - kernel_col + 1
    img = np.zeros(shape=(output_row, output_col) )
    
    for i in range(output_row):
        for j in range(output_col):
            img[i, j] = np.sum( image[i:i + kernel_row, j: j + kernel_col] * kernel )

    return img
        

def full_padding(image, kernel):

    row, col = image.shape
    kernel_row, kernel_col = kernel.shape

    output_row, output_col = row + ( row - kernel_row - 1 ) * 2, col + ( col - kernel_col - 1 ) * 2
    img = np.zeros(shape=(output_row, output_col) )
    ans = np.zeros(shape=(row, col))
    
    for i in range(row):
        for j in range(col):
            img[i + row - kernel_row - 1, j + col - kernel_col - 1] = image[i, j]
            ans[i, j] = np.sum( img[i: i + kernel_row, j: j + kernel_col] * kernel )
            
    return img, ans

In [3]:
import math

def stride(image, stride=(1, 1)):

    img = np.array([])
    row, col = image.shape
    stride_row, stride_col = stride

    for i in range(0, row, stride_row):
        for j in range(0, col, stride_col):
            img = np.append(img, image[i, j] )

    img = img.reshape( math.ceil(row/stride_row), math.ceil(col/stride_col) )
    
    return img

In [4]:
def average_pooling(image, pool, mode='global'):

    if mode == 'global':
        return np.array([np.mean(image)])
    
    row, col = image.shape
    pool_row, pool_col = pool

    output_row, output_col = row - pool_row + 1, col - pool_col + 1
    img = np.zeros(shape=(output_row, output_col) )
    
    for i in range(output_row):
        for j in range(output_col):
            img[i, j] = np.mean( image[ i: i + pool_row, j: j + pool_col ] )

    return img


def max_pooling(image, pool, mode='global'):
    
    if mode == 'global':
        return np.array([np.max(image)])
        
    row, col = image.shape
    pool_row, pool_col = pool

    output_row, output_col = row - pool_row + 1, col - pool_col + 1
    img = np.zeros(shape=(output_row, output_col) )

    for i in range(output_row):
        for j in range(output_col):
            img[i, j] = np.max( image[i: i + pool_row, j : j + pool_col ] )

    return img

In [5]:
image = np.array([
    [0, 3, 0, 0],
    [0, 1, 0, 2],
    [0, 2, 0, 0],
    [0, 1, 0, 0],
],)

kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

average_pooling(image, kernel.shape), max_pooling(image, kernel.shape), average_pooling(image, kernel.shape, 'local'), max_pooling(image, kernel.shape, 'local')

(array([0.5625]),
 array([3]),
 array([[0.66666667, 0.88888889],
        [0.44444444, 0.66666667]]),
 array([[3., 3.],
        [2., 2.]]))

In [6]:
image = np.array([
    [1, 1, 1, 2, 5, 3],
    [1, 3, 4, 2, 5, 1],
    [1, 4, 2, 1, 6, 4],
    [1, 2, 4, 5, 6, 1],
    [1, 3, 2, 1, 2, 4],
    [1, 3, 5, 1, 4, 4],
])

kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

same_padding(image, kernel), valid_padding(image, kernel), full_padding(image, kernel)

((array([[0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 1., 1., 1., 2., 5., 3., 0.],
         [0., 1., 3., 4., 2., 5., 1., 0.],
         [0., 1., 4., 2., 1., 6., 4., 0.],
         [0., 1., 2., 4., 5., 6., 1., 0.],
         [0., 1., 3., 2., 1., 2., 4., 0.],
         [0., 1., 3., 5., 1., 4., 4., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.]]),
  array([[-4., -3.,  0., -5.,  0., 10.],
         [-8., -4.,  3., -9., -3., 16.],
         [-9., -7.,  1., -7.,  2., 17.],
         [-9., -5.,  2., -6., -2., 14.],
         [-8., -8.,  1., -1., -2., 12.],
         [-6., -5.,  4.,  1., -6.,  6.]])),
 array([[-4.,  3., -9., -3.],
        [-7.,  1., -7.,  2.],
        [-5.,  2., -6., -2.],
        [-8.,  1., -1., -2.]]),
 (array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 1., 1., 1., 2., 5., 3., 0., 0.],
         [0., 0., 1., 3., 4., 2., 5., 1., 0., 0.],
         [0., 0., 1., 4., 2., 1., 6., 4., 0., 0.],
         [0., 0., 1., 2., 4.

In [8]:
image = np.array([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25],
    [26, 27, 28, 29, 30]
])

_stride = (3, 3)
img = stride(image, _stride)
img

array([[ 1.,  4.],
       [16., 19.]])